In [6]:
import re, warnings
import numpy as np
from pathlib import Path
from transformers import pipeline

warnings.filterwarnings("ignore")

In [2]:
PATRONES_PROHIBIDOS = [
    # Sexual
    r"\b(sexo|sexual|pornografía|porno|desnudo|masturba[rc]|violación|violador|orgasmo|erección|eyaculación)\b",
    # Violencia
    r"\b(matar|asesinar|apuñalar|violento|sangre|sádico|violar|golpear|estrangular|arma|navaja|tiroteo)\b",
    # Lenguaje obsceno
    r"\b(estúpido|idiota|imbécil|mierda|maldito|puta|pendejo|jódete|cabr[oó]n|zorra|cul[oó]|verga|chingar)\b",
    # Discriminación racial
    r"\b(negro de mierda|maldito negro|judío maldito|indio bruto|chino de mierda|sudaca|nazi)\b",
    # Discriminación de género e identidad
    r"\b(maric[oó]n|puto|lesbiana de mierda|transexual deforme|machorra|hembra estúpida|feminazi|gay asqueroso)\b"
]

def contiene_lenguaje_ofensivo_regex(texto: str) -> bool:
    texto = texto.lower()
    for patron in PATRONES_PROHIBIDOS:
        if re.search(patron, texto):
            return True
    return False

In [3]:
try:
    moderador_hf = pipeline("text-classification", model="unitary/toxic-bert", top_k=None)
except Exception as e:
    print("❌ Error al cargar modelo moderador HF:", e)
    moderador_hf = None

def contiene_lenguaje_ofensivo_modelo(texto: str, umbral=0.6) -> bool:
    if not moderador_hf:
        return False  # fallback si el modelo no cargó
    resultado = moderador_hf(texto)
    for pred in resultado[0]:
        if pred["label"] != "non-toxic" and pred["score"] > umbral:
            return True
    return False

def is_inappropriate_input(texto: str) -> bool:
    return (
        contiene_lenguaje_ofensivo_regex(texto)
        or contiene_lenguaje_ofensivo_modelo(texto)
    )    

In [4]:
frases_prueba = [
    "Hola, ¿me puedes ayudar con mi subsidio de lentes?",
    "Eres un idiota, no sabes nada.",
    "Quiero matar a alguien.",
    "Me gustaría saber si tengo derecho a compensación.",
    "Esto es una mierda, no sirve para nada.",
    "Mi cédula es 1098765432",
    "Vamos a tener sexo explícito.",
    "Gracias por tu ayuda tan profesional.",
    "Maldita sea, me engañaron.",
    "No me importa tu raza o religión.",
]

In [5]:
for frase in frases_prueba:
    ofensivo = is_inappropriate_input(frase)
    estado = "❌ Inapropiado" if ofensivo else "✅ Aceptado"
    print(f"🗨️ Entrada: {frase}\n🔎 Resultado: {estado}\n{'-' * 60}")

🗨️ Entrada: Hola, ¿me puedes ayudar con mi subsidio de lentes?
🔎 Resultado: ✅ Aceptado
------------------------------------------------------------
🗨️ Entrada: Eres un idiota, no sabes nada.
🔎 Resultado: ❌ Inapropiado
------------------------------------------------------------
🗨️ Entrada: Quiero matar a alguien.
🔎 Resultado: ❌ Inapropiado
------------------------------------------------------------
🗨️ Entrada: Me gustaría saber si tengo derecho a compensación.
🔎 Resultado: ✅ Aceptado
------------------------------------------------------------
🗨️ Entrada: Esto es una mierda, no sirve para nada.
🔎 Resultado: ❌ Inapropiado
------------------------------------------------------------
🗨️ Entrada: Mi cédula es 1098765432
🔎 Resultado: ✅ Aceptado
------------------------------------------------------------
🗨️ Entrada: Vamos a tener sexo explícito.
🔎 Resultado: ❌ Inapropiado
------------------------------------------------------------
🗨️ Entrada: Gracias por tu ayuda tan profesional.
🔎 Result